# Steerable Tetris: Group Convolution in the Irrep Basis

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This colab demonstrates the equivalence between regular-rep group convolution and steerable convolution (irrep basis). We:

1. Train a regular-rep $D_4$ group conv model on tetris classification
2. Compute the canonical irrep decomposition ($Q_G$, spatial basis functions)
3. Show steerable conv as a wrapper: $Q_G \cdot \text{regular\_conv} \cdot Q_G^{-1}$
4. Implement norm-based equivariant activations
5. Build and train a steerable tetris classifier
6. Verify equivalence between the two approaches

In [ ]:
%pip install -q pymatgen symm4ml

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from IPython.display import HTML

from symm4ml import groups, rep, linalg, plot
from symm4ml.group_conv import (
    apply_group_element_2D,
    orbit_2D,
    image2D_permutation_representation,
    image2D_group_convolution_filter_bank,
    image2D_group_convolution,
)
from symm4ml.steerable import (
    irrep_decomposition,
    spatial_basis_functions,
    canonicalize_irreps,
    norm_activation,
    regular_to_steerable,
    steerable_to_regular,
    wrapper_steerable_conv,
    decompose_into_irreps,
)
from symm4ml.utils import match_character_tables, print_character_table

torch.manual_seed(42)
np.random.seed(42)

---
## 1. Setup: $D_4$ Group and Tetris Pieces

Same setup as the group convolution tetris colab.

In [ ]:
# Build D4 group
rot90 = np.array([[np.cos(np.pi/2), -np.sin(np.pi/2)],
                  [np.sin(np.pi/2),  np.cos(np.pi/2)]])
sigma_v = np.array([[-1., 0.], [0., 1.]])

D4 = np.array(groups.generate_group(np.array([sigma_v, rot90]))[::-1])  # E first
D4_torch = torch.tensor(D4, dtype=torch.float32)
D4_table = groups.make_multiplication_table(D4)
D4_reg_rep = torch.tensor(rep.regular_representation(D4_table), dtype=torch.float32)
D4_perm_rep_3x3 = image2D_permutation_representation(D4_torch, [3, 3])

print(f"D4: {len(D4)} elements")
print(f"Regular rep shape: {D4_reg_rep.shape}")
print(f"Permutation rep (3×3 kernel) shape: {D4_perm_rep_3x3.shape}")

In [ ]:
# Tetris pieces (same definitions as group_conv_tetris.ipynb)
PIECES = {
    'I': np.array([[-2, 0], [-1, 0], [0, 0], [1, 0]]),
    'O': np.array([[0, 0], [0, 1], [1, 0], [1, 1]]),
    'T': np.array([[0, -1], [0, 0], [0, 1], [1, 0]]),
    'S': np.array([[0, 0], [0, 1], [1, -1], [1, 0]]),
    'Z': np.array([[0, -1], [0, 0], [1, 0], [1, 1]]),
    'L': np.array([[-1, 0], [0, 0], [1, 0], [1, 1]]),
    'J': np.array([[-1, 0], [0, 0], [1, 0], [1, -1]]),
}
PIECE_NAMES = list(PIECES.keys())
GRID_SIZE = 9
CENTER = GRID_SIZE // 2


def place_piece(coords, grid_size=9, translation=None):
    """Place piece on a grid."""
    min_r, max_r = coords[:, 0].min(), coords[:, 0].max()
    min_c, max_c = coords[:, 1].min(), coords[:, 1].max()
    if translation is None:
        tr = np.random.randint(-min_r, grid_size - max_r)
        tc = np.random.randint(-min_c, grid_size - max_c)
    else:
        tr, tc = translation
    img = np.zeros((grid_size, grid_size))
    for r, c in coords:
        img[r + tr, c + tc] = 1.0
    return img


# D4: 5 classes (L↔J, S↔Z equivalent under reflections)
D4_CLASSES = {'I': 0, 'O': 1, 'T': 2, 'S': 3, 'Z': 3, 'L': 4, 'J': 4}
D4_CLASS_NAMES = ['I', 'O', 'T', 'S/Z', 'L/J']
NUM_CLASSES = 5

In [ ]:
# Create training and test sets
def make_training_set(class_map):
    """One centered image per class."""
    seen_classes = set()
    piece_names_to_use = []
    for name in PIECE_NAMES:
        if class_map[name] not in seen_classes:
            seen_classes.add(class_map[name])
            piece_names_to_use.append(name)
    imgs = [place_piece(PIECES[name], GRID_SIZE, translation=(CENTER, CENTER))
            for name in piece_names_to_use]
    labels = [class_map[name] for name in piece_names_to_use]
    return (torch.tensor(np.array(imgs), dtype=torch.float32).unsqueeze(1),
            torch.tensor(labels, dtype=torch.long),
            piece_names_to_use)


def make_test_set(class_map, group_matrices, n_per_piece=100):
    """Random orientations and random positions."""
    imgs, labels = [], []
    for name, coords in PIECES.items():
        for _ in range(n_per_piece):
            g_idx = np.random.randint(len(group_matrices))
            rotated = apply_group_element_2D(coords, group_matrices[g_idx])
            imgs.append(place_piece(rotated, GRID_SIZE))
            labels.append(class_map[name])
    return (torch.tensor(np.array(imgs), dtype=torch.float32).unsqueeze(1),
            torch.tensor(labels, dtype=torch.long))


train_imgs, train_labels, train_pieces = make_training_set(D4_CLASSES)
test_imgs, test_labels = make_test_set(D4_CLASSES, D4)
print(f"Training: {len(train_labels)} images ({train_pieces})")
print(f"Test: {len(test_labels)} images (random D4 transforms + positions)")

---
## 2. Regular-Rep Group Conv Model (Baseline)

We first train the standard $D_4$ group conv model from the tetris colab. This serves as our baseline — we'll later show that steerable conv produces identical results in a different basis.

In [ ]:
def lifting_convolution(perm_rep, x, filter_weights):
    """Lift a scalar image to a feature map over the group.
    x: [batch, c_in, H, W] -> [batch, c_out, |G|, H, W]
    """
    z, c, h, w = x.shape
    d, c_f, kh, kw = filter_weights.shape
    k = perm_rep.shape[0]
    f_expanded = filter_weights.unsqueeze(2)
    fb = image2D_group_convolution_filter_bank(perm_rep, f_expanded)
    fb = fb[:, :, :, 0, :, :]
    fb_flat = fb.reshape(k * d, c_f, kh, kw)
    out = F.conv2d(x, fb_flat, padding=kh // 2)
    return out.reshape(z, k, d, h, w).permute(0, 2, 1, 3, 4)


class GroupEquivariantCNN(torch.nn.Module):
    """D4 group-equivariant CNN: lifting conv -> group conv -> pool -> classify."""

    def __init__(self, num_classes=5, c1=16, c2=32):
        super().__init__()
        self.register_buffer('perm_rep', D4_perm_rep_3x3)
        self.register_buffer('reg_rep', D4_reg_rep)
        self.group_order = 8
        self.lift_w = torch.nn.Parameter(torch.randn(c1, 1, 3, 3) * 0.3)
        self.conv_w = torch.nn.Parameter(torch.randn(c2, c1, 8, 3, 3) * 0.1)
        self.fc = torch.nn.Linear(c2, num_classes)

    def forward(self, x):
        x = lifting_convolution(self.perm_rep, x, self.lift_w)
        x = torch.relu(x)
        x = image2D_group_convolution(self.perm_rep, self.reg_rep, x, self.conv_w)
        x = torch.relu(x)
        x = x.mean(dim=(-2, -1))  # spatial pool -> [batch, c2, |G|]
        x = x.mean(dim=-1)        # group pool  -> [batch, c2]
        return self.fc(x)

In [ ]:
def train_model(model, train_imgs, train_labels, test_imgs, test_labels,
                epochs=500, lr=1e-3):
    """Train and return loss/accuracy history."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = torch.nn.CrossEntropyLoss()
    history = {'loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        model.train()
        logits = model(train_imgs)
        loss = criterion(logits, train_labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            train_acc = (model(train_imgs).argmax(1) == train_labels).float().mean().item()
            test_acc = (model(test_imgs).argmax(1) == test_labels).float().mean().item()
        history['loss'].append(loss.item())
        history['train_acc'].append(train_acc)
        history['test_acc'].append(test_acc)

        if (epoch + 1) % 100 == 0:
            print(f"Epoch {epoch+1:3d}: loss={loss.item():.4f}  "
                  f"train_acc={train_acc:.3f}  test_acc={test_acc:.3f}")
    return history


def plot_history(history, title, n_train=None):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax1.plot(history['loss']); ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
    ax1.set_title('Training loss')
    train_label = f'Train ({n_train} centered imgs)' if n_train else 'Train'
    ax2.plot(history['train_acc'], label=train_label)
    ax2.plot(history['test_acc'], label='Test (random rot + pos)')
    ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy'); ax2.legend(); ax2.set_ylim(0, 1.05)
    ax2.set_title('Accuracy')
    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Train the regular-rep baseline
torch.manual_seed(0)
reg_model = GroupEquivariantCNN(num_classes=NUM_CLASSES)
reg_history = train_model(reg_model, train_imgs, train_labels,
                          test_imgs, test_labels, epochs=200)
plot_history(reg_history,
             'D₄ Regular-Rep CNN: 5 training images → test on all transforms + positions',
             n_train=len(train_labels))

---
## 3. Canonical Irrep Decomposition

The steerable approach works in the **irrep basis** instead of the regular-rep basis. We need:

1. **$Q_G$**: the change-of-basis matrix that block-diagonalizes the regular rep into irreps
2. **Canonical irreps**: irrep matrices pinned to geometric (x, y) directions
3. **Spatial basis functions**: kernel patterns that transform as each irrep

The key tool is `steerable.irrep_decomposition()`, which uses `match_character_tables` to identify irreps in standard order (A₁, A₂, B₁, B₂, E) and `canonicalize_irreps` to fix the gauge of the 2D E irrep.

In [ ]:
# Compute canonical irrep decomposition
basis = irrep_decomposition(D4_table, vec_rep=D4)

print("D₄ irrep decomposition:")
print(f"  Labels: {basis.labels}")
print(f"  Irrep slices in Q_G:")
for label, (s, e, d) in zip(basis.labels, basis.irrep_slices):
    print(f"    {label}: rows {s}:{e}  (dim {d})")
print(f"\n  Q_G shape: {basis.Q_G.shape}")
print(f"  Regular rep decomposes as: {' ⊕ '.join(basis.labels)}")
print(f"  Dimensions: {' + '.join(str(d) for _, _, d in basis.irrep_slices)} = {sum(d for _, _, d in basis.irrep_slices)}")

In [ ]:
# Verify: Q_G block-diagonalizes the regular rep
Q_G = torch.tensor(basis.Q_G, dtype=torch.float32)
Q_G_inv = torch.linalg.inv(Q_G)

print("Verifying Q_G block-diagonalizes D_reg(g) for all g ∈ D₄:")
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for g in range(8):
    ax = axes[g // 4, g % 4]
    block_diag = Q_G @ D4_reg_rep[g] @ Q_G_inv
    ax.imshow(block_diag.numpy(), cmap='RdBu', vmin=-1, vmax=1, interpolation='nearest')
    ax.set_title(f'g{g}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

    # Draw irrep block boundaries
    for s, e, d in basis.irrep_slices:
        rect = plt.Rectangle((s - 0.5, s - 0.5), d, d, fill=False,
                              edgecolor='black', linewidth=1.5)
        ax.add_patch(rect)

fig.suptitle('Q_G · D_reg(g) · Q_G⁻¹  (should be block-diagonal)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Spatial Basis Functions

For a 3×3 kernel, the spatial basis functions $B^\alpha_m(\mathbf{y})$ are pixel patterns that transform as each irrep. These are the "steerable filters" — fixed by group theory, with only scalar weights learned.

In [ ]:
# Compute spatial basis functions for 3×3 kernel
sp_basis = spatial_basis_functions(D4_perm_rep_3x3.numpy(), basis.irreps)

# The unique irrep labels (before multiplicity expansion in reg rep)
unique_labels = ['A1', 'A2', 'B1', 'B2', 'E']

print("Spatial basis functions for 3×3 kernel under D₄:")
for i, (label, sb) in enumerate(zip(unique_labels, sp_basis)):
    n_copies = sb.shape[0]
    dim = sb.shape[1]
    print(f"  {label}: {n_copies} copies × dim {dim} = {n_copies * dim} functions")
print(f"  Total: {sum(sb.shape[0] * sb.shape[1] for sb in sp_basis)} = 9 pixels ✓")

# Visualize spatial basis functions
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
plot_idx = 0
for i, (label, sb) in enumerate(zip(unique_labels, sp_basis)):
    for copy in range(sb.shape[0]):
        for row in range(sb.shape[1]):
            if plot_idx >= 10:
                break
            ax = axes[plot_idx // 5, plot_idx % 5]
            grid = sb[copy, row].reshape(3, 3)
            im = ax.imshow(grid, cmap='RdBu', vmin=-0.6, vmax=0.6, interpolation='nearest')
            suffix = f' (copy {copy+1})' if sb.shape[0] > 1 else ''
            row_label = f', row {row+1}' if sb.shape[1] > 1 else ''
            ax.set_title(f'{label}{suffix}{row_label}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])
            plot_idx += 1

for j in range(plot_idx, 10):
    axes[j // 5, j % 5].axis('off')

fig.suptitle('Spatial basis functions B^α_m(y) for 3×3 kernel under D₄', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Steerable Conv as Wrapper

The key pedagogical insight: **steerable convolution is just regular convolution in a different basis.** Concretely:

$$\text{steerable\_conv}(\hat{x}) = Q_G \cdot \text{regular\_conv}(Q_G^{-1} \hat{x}) \cdot Q_G^{-1}$$

where $\hat{x} = Q_G \, x$ are the features in the irrep basis. Same weights, same computation, different coordinate system.

Let's verify this by taking the trained regular-rep model and applying it through the $Q_G$ lens.

In [ ]:
# Take the trained model and compare regular vs steerable features
reg_model.eval()
with torch.no_grad():
    # Run lifting conv in regular rep
    x = train_imgs[:1]  # single test image
    features_reg = lifting_convolution(D4_perm_rep_3x3, x, reg_model.lift_w)
    print(f"Regular-rep features after lifting: {features_reg.shape}")
    # [1, c1, 8, H, W]

    # Convert to irrep basis
    features_irrep = regular_to_steerable(features_reg, Q_G)
    print(f"Irrep-basis features after Q_G: {features_irrep.shape}")

    # Convert back — should match original
    features_back = steerable_to_regular(features_irrep, Q_G)
    diff = (features_reg - features_back).abs().max().item()
    print(f"Round-trip error (reg → irrep → reg): {diff:.2e}")

    # Apply group conv in both bases and compare
    conv_reg = image2D_group_convolution(D4_perm_rep_3x3, D4_reg_rep,
                                         features_reg, reg_model.conv_w)
    conv_steerable = wrapper_steerable_conv(Q_G, D4_perm_rep_3x3, D4_reg_rep,
                                            features_irrep, reg_model.conv_w)
    # Convert steerable result back to regular for comparison
    conv_steerable_as_reg = steerable_to_regular(conv_steerable, Q_G)
    conv_diff = (conv_reg - conv_steerable_as_reg).abs().max().item()
    print(f"\nGroup conv output difference (regular vs wrapper steerable): {conv_diff:.2e}")
    print("✓ Steerable conv is exactly equivalent to regular conv!" if conv_diff < 1e-4
          else "✗ Difference too large")

In [ ]:
# Visualize: what do features look like in the irrep basis?
reg_model.eval()
with torch.no_grad():
    x = train_imgs[:1]
    feat_reg = lifting_convolution(D4_perm_rep_3x3, x, reg_model.lift_w)
    feat_irrep = regular_to_steerable(feat_reg, Q_G)

channel = 0  # look at first channel

fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))

# Top row: regular-rep features (one image per group element)
for g in range(8):
    ax = axes[0, g]
    ax.imshow(feat_reg[0, channel, g].numpy(), cmap='RdBu', interpolation='nearest')
    ax.set_title(f'g{g}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
axes[0, 0].set_ylabel('Regular rep', fontsize=10)

# Bottom row: irrep-basis features (one image per irrep component)
irrep_labels_expanded = []
for label, (s, e, d) in zip(basis.labels, basis.irrep_slices):
    for j in range(d):
        irrep_labels_expanded.append(f'{label}[{j}]' if d > 1 else label)

for i in range(8):
    ax = axes[1, i]
    ax.imshow(feat_irrep[0, channel, i].numpy(), cmap='RdBu', interpolation='nearest')
    ax.set_title(irrep_labels_expanded[i], fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
axes[1, 0].set_ylabel('Irrep basis', fontsize=10)

fig.suptitle(f'Feature maps (channel {channel}): regular rep vs irrep basis',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. Steerable CNN with Norm-Based Activations

Now we build a model that works entirely in the irrep basis. The key difference: we can't use pointwise ReLU (it breaks equivariance for non-permutation representations). Instead, we use **norm-based activations**:

$$\mathbf{x}_\alpha \;\mapsto\; \frac{\text{ReLU}(\|\mathbf{x}_\alpha\|)}{\|\mathbf{x}_\alpha\|} \cdot \mathbf{x}_\alpha$$

For 1D irreps this reduces to $x \mapsto \text{ReLU}(|x|) \cdot \text{sign}(x)$. For the trivial irrep A₁, it's just ReLU.

This model uses the **wrapper** approach: steerable conv = $Q_G \cdot \text{regular\_conv} \cdot Q_G^{-1}$, with norm activations in the irrep basis.

In [ ]:
# Demonstrate norm activation
print("Norm-based activation preserves equivariance:")
print("For each irrep block, we compute the norm, apply ReLU, and rescale.\n")

# Show what happens for a 1D irrep (B1) under sign flip
x_1d = torch.tensor([[[[-1.5]]]])  # negative value
print(f"B1 feature: x = {x_1d.item():.1f}")
print(f"  ReLU(x) = {torch.relu(x_1d).item():.1f}  ← breaks equivariance (loses sign)")
print(f"  norm_act(x) = ReLU(|x|)·sign(x) = {torch.relu(torch.abs(x_1d)).item() * torch.sign(x_1d).item():.1f}  ← preserves sign")

# Show for a 2D irrep (E) under rotation
x_2d = torch.tensor([1.0, -1.0])
rot = torch.tensor([[0., -1.], [1., 0.]])  # 90° rotation
x_rot = rot @ x_2d

norm_x = x_2d.norm()
norm_x_rot = x_rot.norm()
print(f"\nE feature: x = {x_2d.numpy()}")
print(f"  D(90°)·x = {x_rot.numpy()}")
print(f"  ||x|| = {norm_x:.3f},  ||D·x|| = {norm_x_rot:.3f}  (same! — norm is invariant)")
print(f"  σ(||x||)/||x|| · x = {(torch.relu(norm_x) / norm_x * x_2d).numpy()}")
print(f"  σ(||D·x||)/||D·x|| · D·x = {(torch.relu(norm_x_rot) / norm_x_rot * x_rot).numpy()}")
print(f"  D · [σ(||x||)/||x|| · x] = {(rot @ (torch.relu(norm_x) / norm_x * x_2d)).numpy()}")
print("  ✓ Last two lines are equal — activation commutes with group action")

In [ ]:
class SteerableCNN(torch.nn.Module):
    """D4 steerable CNN using wrapper approach.

    Architecture: lifting conv → norm activation → group conv → norm activation → pool → classify

    Works entirely in the irrep basis. The convolutions use Q_G to convert
    to/from regular rep (wrapper approach), but activations and pooling
    are done natively in the irrep basis.
    """

    def __init__(self, num_classes=5, c1=16, c2=32):
        super().__init__()
        self.register_buffer('perm_rep', D4_perm_rep_3x3)
        self.register_buffer('reg_rep', D4_reg_rep)
        self.register_buffer('Q_G', Q_G)
        self.group_order = 8
        self.irrep_slices = basis.irrep_slices

        # Same parameter shapes as regular-rep model
        self.lift_w = torch.nn.Parameter(torch.randn(c1, 1, 3, 3) * 0.3)
        self.conv_w = torch.nn.Parameter(torch.randn(c2, c1, 8, 3, 3) * 0.1)
        self.fc = torch.nn.Linear(c2, num_classes)

    def forward(self, x):
        # Lifting conv (produces regular-rep features, then convert to irrep basis)
        x = lifting_convolution(self.perm_rep, x, self.lift_w)
        x = regular_to_steerable(x, self.Q_G)

        # Norm-based activation in irrep basis
        x = norm_activation(x, self.irrep_slices)

        # Group conv via wrapper: Q_G @ reg_conv @ Q_G^{-1}
        x = wrapper_steerable_conv(self.Q_G, self.perm_rep, self.reg_rep,
                                   x, self.conv_w)

        # Norm-based activation again
        x = norm_activation(x, self.irrep_slices)

        # Pool: spatial average
        x = x.mean(dim=(-2, -1))  # [batch, c2, 8]

        # Invariant output: project to A1 (trivial irrep = first component)
        # A1 slice is (0, 1, 1) — the first row of Q_G
        a1_start, a1_end, _ = self.irrep_slices[0]
        x = x[:, :, a1_start:a1_end].squeeze(-1)  # [batch, c2]

        return self.fc(x)

In [ ]:
# Train the steerable model
torch.manual_seed(0)
steer_model = SteerableCNN(num_classes=NUM_CLASSES)
steer_history = train_model(steer_model, train_imgs, train_labels,
                            test_imgs, test_labels, epochs=200)
plot_history(steer_history,
             'D₄ Steerable CNN (norm activations): 5 training images → test on all transforms',
             n_train=len(train_labels))

---
## 6. Verify Equivariance

Both models should produce identical logits for all orientations of the same piece. Let's verify.

In [ ]:
def verify_equivariance(model, model_name, group_matrices, class_names, grid_size=9):
    """Verify that model gives identical logits for all orientations."""
    model.eval()
    perm_rep_full = image2D_permutation_representation(
        torch.tensor(group_matrices, dtype=torch.float32), [grid_size, grid_size]
    )
    test_pieces = ['T', 'L', 'S']
    n_g = len(group_matrices)

    fig, axes = plt.subplots(len(test_pieces), n_g + 1, figsize=(2*(n_g+1), 2.5*len(test_pieces)))
    for row, name in enumerate(test_pieces):
        coords = PIECES[name]
        img = place_piece(coords, grid_size, translation=(CENTER, CENTER))
        img_t = torch.tensor(img, dtype=torch.float32).reshape(1, 1, grid_size, grid_size)
        with torch.no_grad():
            logits_orig = model(img_t)
            pred = class_names[logits_orig.argmax(1).item()]

        ax = axes[row, 0]
        ax.imshow(img, cmap='Purples', vmin=0, vmax=1.5, interpolation='nearest')
        ax.set_title(f'{name} → {pred}', fontsize=9, fontweight='bold')
        ax.set_xticks([]); ax.set_yticks([])

        for g_idx in range(n_g):
            rot_flat = perm_rep_full[g_idx] @ torch.tensor(img.flatten(), dtype=torch.float32)
            rot_img = rot_flat.reshape(grid_size, grid_size)
            with torch.no_grad():
                logits_rot = model(rot_img.reshape(1, 1, grid_size, grid_size))
                max_diff = (logits_rot - logits_orig).abs().max().item()
            ax = axes[row, g_idx + 1]
            ax.imshow(rot_img.numpy(), cmap='Purples', vmin=0, vmax=1.5, interpolation='nearest')
            mark = '✓' if max_diff < 1e-4 else f'Δ={max_diff:.1e}'
            ax.set_title(f'g{g_idx} {mark}', fontsize=9)
            ax.set_xticks([]); ax.set_yticks([])

    fig.suptitle(f'{model_name}: logits identical for all D₄ orientations',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


verify_equivariance(reg_model, 'Regular-rep CNN', D4, D4_CLASS_NAMES)
verify_equivariance(steer_model, 'Steerable CNN', D4, D4_CLASS_NAMES)

---
## 7. Comparison and Summary

| | **Regular-Rep CNN** | **Steerable CNN** |
|---|---|---|
| **Basis** | Group elements $\\{e, r, r^2, \\ldots, s, sr, \\ldots\\}$ | Irreps $A_1 \\oplus A_2 \\oplus B_1 \\oplus B_2 \\oplus E \\oplus E$ |
| **Nonlinearity** | Pointwise ReLU (works because $D^{\\text{reg}}$ permutes) | Norm-based (preserves irrep structure) |
| **Parameters** | Same | Same (wrapper uses identical weights) |
| **Outputs** | Identical | Identical (up to basis change $Q_G$) |
| **Invariant pool** | Average over group dim | Project to $A_1$ slice |
| **Scalability** | Requires $|G|$ copies of every feature | Can use arbitrary irrep content per layer |

**Key takeaway**: Steerable convolution is not a new operation — it's the same group convolution viewed through the irrep decomposition. For finite groups like $D_4$, the regular-rep approach is simpler and works fine. The steerable approach becomes essential for **continuous groups** like $\\text{SO}(3)$, where the regular representation is infinite-dimensional but we can work with finitely many irreps (spherical harmonics).

The **norm-based activation** is the price of working in the irrep basis: we can't use arbitrary pointwise nonlinearities, but the constrained activations are what enforce equivariance in a basis where the group action is no longer a simple permutation.